# Binary classification for 'distributed system' related terminology, using Word to Vector and Back Propagation.

## Summary

##### To create this model, Word to Vector (W2V) was used to interrogate and extract a dataset capable of training a Back Propagation model, to classify words as either "related" or "not related" to the concept of a "distributed system"

##### The W2V model was trained on text from the book "DISTRIBUTED SYSTEMS Fourth edition Version 4.03, January 2025", by Maarten van Steen Andrew S. Tanenbaum.

##### This text was chosen because the probability it would contain language strongly associated with the concept of a "distributed system" within context, was high. The expectation was that a strong output of words most similar to "distributed" and "system" would be produced by the W2V model. That list would then serve as a good foundation for words related to "distributed" and "system" for Back Propagation training.

## Library Setup

##### In this section, the libraries required to peform W2V and back propagation are installed. Gensim was used for the W2V model; SpaCy and Tensorflow for the back propagation. Text processing tools are loaded to clean and prepare the data to be processed by the model and reduce errors. For this we use a lemmatizer and tokenise the sentences.

In [1]:
# Install the libraries we need (only needs to run once)
%pip install --upgrade gensim
%pip install spacy pandas tensorflow numpy seaborn scikit-learnnltk
!pip install nbformat>=4.2.0
!python -m spacy download en_core_web_md

# --- Core libraries ---
import gensim                        # The main library for word2vec
import gensim.downloader as api      # For downloading pre-trained models
import tensorflow as tf
from tensorflow.keras import layers, models
import spacy
import sys
import subprocess
import pandas as pd                  # For organizing data in tables
import numpy as np                   # For numerical operations
import re
import nltk                          # Natural Language Toolkit - text processing
nltk.download('punkt_tab')           # Gives lists of stopwords for NLTK
nltk.download('wordnet')             # Linguistic annotated database

# --- Text preprocessing tools ---
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()     # Converts words to base form (e.g., "running" → "run")
from nltk import word_tokenize       # Splits text into individual words
nltk.download('stopwords')           # Download list of common words to ignore
from nltk.corpus import stopwords
stops = set(stopwords.words('english'))  # Words like "the", "is", "at" that don't carry meaning are removed
import string
punct = list(string.punctuation)     # Punctuation marks (: . , ! ? etc) are removed.

# --- Visualization // analysis // testing ---
from sklearn.model_selection import train_test_split # Splits data into training and test samples
from sklearn.metrics import (precision_score, f1_score, recall_score, # Metrics and visualization
                             accuracy_score, classification_report, 
                             confusion_matrix)



# Installs further  models
print("Installing the required spacy model...")
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_md"])

# loads the SpaCy model
nlp = spacy.load("en_core_web_md")  # Using medium-sized English model with word vectors

print("\n           --- Complete ---             ")
print(f"\nLoaded! Vocabulary size: {len(nlp.vocab):,} entries")
print(f"Vector dimensions: {nlp.vocab.vectors.shape[1]}")
print(f"Words with vectors: {nlp.vocab.vectors.shape[0]:,}")

Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement scikit-learnnltk (from versions: none)
ERROR: No matching distribution found for scikit-learnnltk
Note: you may need to restart the kernel to use updated packages.
zsh:1: 4.2.0 not found
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.8.0/en_core_web_md-3.8.0-py3-none-any.whl (33.5 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/nicolasharper/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/nicolasharper/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nicolasharper/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Installing the required spacy model...
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.8.0/en_core_web_md-3.8.0-py3-none-any.whl (33.5 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')

           --- Complete ---             

Loaded! Vocabulary size: 764 entries
Vector dimensions: 300
Words with vectors: 20,000


## Word to Vector Model

#### Data Preparation

##### For the data to work effectively within the model I needs to be standardised in the following steps:
##### 1.	The text is loaded in .txt format and cleaned. 

##### 2.	The text is split into sentences and placed in a list so the W2V model learns from the sentences which words appear near each other.

##### 3.	The words in each sentence are then tokenized or split into individual words.

##### 4.	The tokenized sentences are the lemmatized, converting words to their base forms, and then go through stop word and punctuation removal. This helps W2V recognize that all the words are related to the same concept and reduce noise.

In [2]:
# Step 1: Load and clean the raw text
# =====================================

# Reads the Distributed Systems 4 text file into memory
with open('Data/Distributed_Systems_4_extracted.txt', 'r') as f:
    distributed_systems_4 = f.read()

# Clean the text:
# Remove non-ASCII characters (special symbols that might cause problems)
# Remove newline characters (\n) and join everything into one string
# Convert to lowercase so "System" and "system" are treated the same
distributed_systems_4 = distributed_systems_4.encode('ascii', 'ignore')  # Keep only standard English characters
distributed_systems_4 = distributed_systems_4.decode()                    # Convert back to regular text
distributed_systems_4 = ' '.join(distributed_systems_4.splitlines())      # Replace newlines with spaces
distributed_systems_4 = distributed_systems_4.lower()                     # Make everything lowercase

# Removes numbers (from the text, pages and contents)

def remove_numbers(text):
    # Base case for strings
    if isinstance(text, str):
        return ''.join(char for char in text if not char.isdigit())
    
    # For lists or other iterables
    elif isinstance(text, list):
        return [remove_numbers(item) for item in text]
    
    # For dictionaries
    elif isinstance(text, dict):
        return {remove_numbers(k): remove_numbers(v) for k, v in text.items()}
    
    # Return unchanged for numbers and other types
    else:
        return text


dsys_nonum = remove_numbers(distributed_systems_4)  # Changed 'text' to 'dsys_nonum'
print("\nComplete")
print(f"Text loaded: {dsys_nonum[:5]}...") 
print(f"Text loaded! First 200 characters: {dsys_nonum[:200]}...")  # Changed 'text' to 'distributed_systems_4'


Complete
Text loaded: distr...
Text loaded! First 200 characters: distributed systems maarten van steen andrew s. tanenbaum th edition version    cover art by max van steen  distributed systems fourth edition version . (january ) maarten van steen andrew s. tanenbau...


In [3]:
# Step 2: Split text into sentences
# ====================================

dsys_clean = dsys_nonum.split('.')                 # Split on periods to get sentences
dsys_clean = [i.strip() for i in dsys_clean]       # Remove extra whitespace from each sentence

print(f"Number of sentences: {len(dsys_clean)}")
print(f"Example sentence: '{dsys_clean[:100]}'")

Number of sentences: 24582
Example sentence: '['distributed systems maarten van steen andrew s', 'tanenbaum th edition version    cover art by max van steen  distributed systems fourth edition version', '(january ) maarten van steen andrew s', 'tanenbaum  copyright   maarten van steen and andrew s', 'tanenbaum published by maarten van steen the first and second edition of this book were previously published by pearson education, inc', 'isbn: ---- (printed version) isbn: ---- (digital version) edition:', 'version:  (february ) all rights to text and illustrations are reserved by maarten van steen and andrew s', 'tanenbaum', 'this work may not be copied, reproduced, or translated in whole or part without written permission of the publisher, except for brief excerpts in reviews or scholarly analysis', 'use with any form of information storage and retrieval, electronic adaptation or whatever, computer software, or by similar or dissimilar methods now known or developed in the future is str

In [4]:
# Step 3: Tokenize into words
# =============================

dsys_tokens = [word_tokenize(i) for i in dsys_clean] # splits the sentences into individual words.

print(f"Example: '{dsys_clean[100]}'")
print(f"Tokenized: {dsys_tokens[100]}")

Example: ''
Tokenized: []


In [5]:
# Step 4: Lemmatization
# =================

dsys_lemmas = [[] for i in range(len(dsys_tokens))]  # Create empty list of tokenized words

for i in range(len(dsys_tokens)):
    for j in dsys_tokens[i]:
        # Only keep words that are NOT stopwords and NOT punctuation
        if j not in stops and j not in punct:
            dsys_lemmas[i].append(lemmatizer.lemmatize(j))

print(f"Original tokens: {dsys_tokens[100]}")
print(f"After cleaning:  {dsys_lemmas[100]}")

Original tokens: []
After cleaning:  []


#### Word2Vec Model Training

##### The model was trained using Gensim W2V and takes three parameters assigning 300 vectors to each word and only learning words appearing more than 20 times. The data is then also formatted using pandas to make it easier to interrogate and export.

In [6]:
# word2vec model training on DS4 text

model = gensim.models.Word2Vec(
    dsys_lemmas,      # Preprocessed sentences
    min_count=20,     # Only learns words appearing 20+ times
    vector_size=300   # Each word gets 300 vectors
)

print("Model trained!")
# Embedding for "system" - these 300 numbers represent the word
# based on how contextual use in the DS4 text
print("The word 'system' is represented by 300 numbers:")
print(model.wv['system'][:10], "... (showing first 10 of 300)")

Model trained!
The word 'system' is represented by 300 numbers:
[ 0.05279703  0.19575776  0.19864784  0.10359232 -0.01777769 -0.11479542
  0.16206048  0.16586153 -0.04181867  0.1767756 ] ... (showing first 10 of 300)


In [7]:
#Data is formated into a table.
vocab_all = model.wv.index_to_key
vectors_all = [model.wv[i] for i in vocab_all]

dsys_all_data = pd.DataFrame(vectors_all)
dsys_all_data.index = vocab_all
print(f"We have {len(vocab_all)} words, each with {len(vectors_all[0])} numbers")
print(f"Example words: {vocab_all[:10]}")

# Find words with similar embeddings (similar contexts in DS4)
# These are words that appear in similar situations to "system" and "distributed"

print("Words most similar to 'system' based on context in Distributed Systems 4:")
model.wv.most_similar(['system'], topn=60) #Finds top 60 words most similar to subject

# Create DataFrame with custom column names
sys_top60 = pd.DataFrame(model.wv.most_similar(['system'], topn=60), columns=['word', 'similarity'])

We have 1340 words, each with 300 numbers
Example words: ['system', 'server', 'process', 'message', 'node', 'client', 'data', 'may', 'example', 'distributed']
Words most similar to 'system' based on context in Distributed Systems 4:


In [8]:
dsys_all_data.head()

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
system,0.052797,0.195758,0.198648,0.103592,-0.017778,-0.114795,0.162060,0.165862,-0.041819,0.176776,...,-0.032868,0.176388,0.067985,0.153138,0.013061,0.275075,0.045275,-0.186363,0.002477,-0.036117
server,0.045460,0.113602,0.087821,0.047140,0.065506,-0.281866,0.185317,0.361488,0.244828,-0.004165,...,0.060966,0.156747,0.286476,-0.010310,0.118715,0.208774,0.047134,-0.062529,-0.008255,0.072917
process,0.081206,0.067021,0.085198,0.094511,0.008781,-0.216882,0.185999,0.341401,0.160790,0.034892,...,-0.009856,0.167342,0.286077,0.005633,0.105814,0.176355,0.061481,-0.077540,-0.000644,0.054180
message,0.057250,0.012989,0.055436,0.117643,0.050363,-0.290239,0.139430,0.385726,0.228404,0.008756,...,0.036977,0.130940,0.293650,-0.020710,0.130559,0.122302,0.025014,-0.045200,0.011072,0.065898
node,0.047996,0.123596,0.139471,0.085637,0.082394,-0.237166,0.189612,0.353795,0.183276,-0.001164,...,0.014630,0.179306,0.221188,0.056666,0.087248,0.165630,0.024539,-0.055832,0.035414,0.046147


In [9]:
sys_top60.head()

,word,similarity
0,distributed,0.976586
1,computer,0.956958
2,computing,0.941663
3,many,0.939750
4,page,0.934098


#### Data Export

##### A block of code added to enable data extraction is .csv format for record keeping.

In [10]:
# Assuming model.wv.most_similar returns a list of tuples (word, similarity)
#use dis_top60 when calculating and extracting words associated with "distributed"
# Change to dis_top60_lst to export .CSV


# Extract the vocabulary (all words) and their embeddings (300 numbers each)
dsys_all_data.to_csv('Data/dsys_all_data.csv', index=True)
print("\nCSV dsys_all_data Output Complete")

sys_top60.to_csv('Data/sys_top60.csv', index=False)
print("\nCSV sys_top60 Output Complete")


CSV dsys_all_data Output Complete

CSV sys_top60 Output Complete


## Back Propagation Model

#### Back Propagation Model Training

##### The back propagation model is then trained using the medium SpaCy NLP model and 205 words, 75 “related” and 130 “not related”. The 75 “related” words were taken from the combined top 60 words produced by the W2V model for “distributed” and “system” with some removed and some added. These words are then assigned a class of “1” and the “not related”, a class of “0”.

In [11]:
# Load the medium language model
nlp = spacy.load("en_core_web_md")

words = [
    # consolidated list of 75 words that are related to "distributed" and "system" (1)
    'distributed', 'computing', 'computer', 'many', 'security', 'screen', 'important', 'often', 'network', 
    'architecture', 'design', 'service', 'operating', 'solution', 'application', 'mechanism', 'protocol', 
    'approach', 'problem', 'conference', 'survey', 'issue', 'algorithm', 'implementation', 'component', 
    'international', 'peer-to-peer', 'discussed', 'mobile', 'decentralized', 'symposium', 'networked', 
    'web', 'performance', 'ieee', 'much', 'internet', 'software', 'organization', 'acm', 'difficult', 
    'virtual', 'discus', 'replication', 'described', 'provided', 'make', 'fault', 'open', 'scheme', 
    'caching', 'principle', 'technique', 'system', 'hub', 'communication', 'developed', 'cloud', 
    'node', 'scalability', 'api', 'authentication', 'synchronization', 'cluster', 'encryption', 
    'decryption', 'middleware', 'interface', 'framework', 'database', 'consensus', 'throughput', 
    'partition', 'quorum', 'sharding',


    # consolidated list of 130 words that have no relation to "distributed" and "system" (0)
    'apple', 'banana', 'orange', 'strawberry', 'pineapple', 'carrot', 'potato', 'onion', 'tomato', 
    'dog', 'cat', 'horse', 'cow', 'bird', 'eagle', 'penguin', 'tree', 'flower', 'grass', 'river', 
    'ocean', 'beach', 'mountain', 'sun', 'moon', 'cloud', 'rain', 'snow', 'pizza', 'burger', 'sushi', 
    'pasta', 'coffee', 'tea', 'milk', 'cake', 'chocolate', 'icecream', 'shirt', 'pants', 'dress', 
    'shoe', 'hat', 'desk', 'chair', 'sofa', 'bed', 'lamp', 'mirror', 'clock', 'fork', 'spoon', 
    'plate', 'bowl', 'canvas', 'brush', 'paint', 'pottery', 'clay', 'sculpture', 'guitar', 'piano', 
    'drum', 'song', 'concert', 'heart', 'lung', 'smile', 'laugh', 'cry', 'love', 'friend', 'family', 
    'baby', 'soccer', 'tennis', 'dance', 'yoga', 'movie', 'actor', 'theater', 'car', 'bus', 'train', 
    'bicycle', 'airplane', 'ticket', 'hotel', 'road', 'bridge', 'red', 'blue', 'green', 'yellow', 
    'circle', 'square', 'triangle', 'big', 'small', 'tall', 'short', 'pillow', 'blanket', 'towel', 
    'soap', 'shampoo', 'toothbrush', 'lipstick', 'comb', 'wallet', 'keys', 'umbrella', 'backpack', 
    'suitcase', 'passport', 'screwdriver', 'camera', 'photograph', 'candle', 'birthday',
    'book', 'newspaper', 'pencil', 'pen', 'eraser', 'page', 'staple', 'sword', 'gun', 'lance'
]

classes = [
    # 1 for words similar to "distributed" and "system"
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    1, 1, 1, 1, 1,

    # 0 for Words that have no relation to "distributed" and "system"
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0
]

vectors = [nlp(i).vector for i in words]

vecs_df = pd.DataFrame(vectors)
vecs_df['word'] = words
vecs_df['label'] = classes
vecs_df = vecs_df.sample(frac = 1)

X = vecs_df.drop(['word', 'label'], axis = 1)
y = vecs_df['label']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Defines the model architecture, 5 nuerons.
model = models.Sequential([
    layers.Dense(5, activation='relu', input_shape=(300,)),  # Hidden layer with 5 neurons
    layers.Dense(1, activation='sigmoid')  # Output layer (binary classification)
])

# Compiles the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

# Train the model
model.fit(X_train, y_train, epochs=30, batch_size=5, verbose=1)

# Makes predictions on the test set
y_pred_prob = model.predict(X_test)  # Get probabilities
y_pred = (y_pred_prob > 0.5).astype(int)  # Convert probabilities to binary labels

# Generate and print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Retrieve weights and biases
for layer in model.layers:
    weights, biases = layer.get_weights()
    print(f"\nLayer: {layer.name}")
    print(f"Weights:\n{weights}")
    print(f"Biases:\n{biases}")

/opt/anaconda3/envs/python3/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 5)              │         1,505 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             6 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,511 (5.90 KB)

 Trainable params: 1,511 (5.90 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 611us/step - accuracy: 0.6014 - loss: 0.6630 
Epoch 2/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 626us/step - accuracy: 0.6643 - loss: 0.5969
Epoch 3/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 615us/step - accuracy: 0.6993 - loss: 0.5292
Epoch 4/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 610us/step - accuracy: 0.7483 - loss: 0.4678
Epoch 5/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 550us/step - accuracy: 0.7692 - loss: 0.4171
Epoch 6/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 533us/step - accuracy: 0.8042 - loss: 0.3755
Epoch 7/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 519us/step - accuracy: 0.8042 - loss: 0.3398
Epoch 8/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 530us/step - accuracy: 0.8951 - loss: 0.3075
Epoch 9/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 538us/step - accuracy: 0.9510 - loss: 0.2649
Epoch 10/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 552us/step - accuracy: 0.9650 - loss: 0.2280
Epoch 11/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 550us/step - accuracy: 0.9650 - loss: 0.2001
Epoch 12/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 539us/ste

#### Model Predictions

In [13]:
# Function that takes a word as an input and predicts "relatedness" to the words it was trained on

def prediction(term):
    word = nlp(term).vector.reshape(1,-1)
    p = model.predict(word)
    if p[0][0] >=0.5:
        return {'related': float(p[0][0])}
    else:
        return {'not related': float(p[0][0])}

In [ ]:
prediction('') #test word goes here